<a href="https://colab.research.google.com/github/BDH-teacher/RL_from_basics/blob/main/RL_from_basic_ch_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DQN 구현 (CartPole)

## 라이브러리 import

In [38]:
# 최신버전으로 수정

!pip install gym pyvirtualdisplay > /dev/null 2>&1
!pip install gymnasium[classic-control] > /dev/null 2>&1

In [39]:
import base64
import collections
import glob
import io
import random

import gymnasium as gym # 최신버전으로 수정
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from IPython import display as ipythondisplay
from IPython.display import HTML

- gym은 openAI GYM 라이브러리 <br/><br/>
- collections 라이브러리는 리플레이 버퍼를 구현할 때 사용
   - deque를 이용하여 **선입선출(first-in-first-out)** 특성을 갖고 있는 리플레이 버퍼를 구현함

## 하이퍼 파라미터 정의

In [40]:
# Hyperparameters

learning_rate = 0.0005
gamma         = 0.98
buffer_limit  = 50000
batch_size    = 32

## Replay Buffer 클래스

In [41]:
class ReplayBuffer():
    def __init__(self):
        self.buffer = collections.deque(maxlen=buffer_limit)

    def put(self, transition):
        self.buffer.append(transition)

    def sample(self, n):
        mini_batch = random.sample(self.buffer, n)
        s_lst, a_lst, r_lst, s_prime_lst, done_mask_lst = [], [], [], [], []

        for transition in mini_batch:
            s, a, r, s_prime, done_mask = transition
            s_lst.append(s)
            a_lst.append([a])
            r_lst.append([r])
            s_prime_lst.append(s_prime)
            done_mask_lst.append([done_mask])

        return torch.tensor(s_lst, dtype=torch.float), torch.tensor(a_lst), \
               torch.tensor(r_lst), torch.tensor(s_prime_lst, dtype=torch.float), \
               torch.tensor(done_mask_lst)

    def size(self):
        return len(self.buffer)

- ReplayBuffer 클래스 정의
   - 최신 데이터를 저장했다가 필요할 때마다 batch_size만큼의 데이터를 뽑아서 제공함 <br/><br/>
   - put 함수 : 데이터를 버퍼에 넣어줌 <br/><br/>
   - sample 함수 : 버퍼에서 랜덤하게 데이터를 뽑아서 미니 배치를 구성함
      - 하나의 데이터는 (s, a, r, s_prime, done_mask)로 구성되어 있음 <br/><br/>
   - done_mask : 종료 상태의 밸류를 마스킹함
      - 종료 상태에서는 0, 나머지상태 1의 값을 가지며, q(s,a)와 곱해져 종료 상태의 밸류를 0으로 만듬 <br/><br/>
   - 이후 s, r 등 요소별로 모아서 pythorch의 텐서로 변환함

## Q밸류 네트워크 클래스

In [42]:
class Qnet(nn.Module):
    def __init__(self):
        super(Qnet, self).__init__()
        self.fc1 = nn.Linear(4, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def sample_action(self, obs, epsilon):
        out = self.forward(obs)
        coin = random.random()
        if coin < epsilon:
            return random.randint(0,1)
        else :
            return out.argmax().item()

- (주의) forward 함수의 마지막 레이어에는 Relu를 사용 하지 않음
   - 맨 마지막 아웃풋은 Q밸류이기 때문에 [-무한, 무한] 사이의 어느 값이든 취할 수 있기 때문에 양수만 리턴하는 Relu를 사용하면 안됨 <br/><br/>
- sampel_action 함수 : 실제로 행할 액션을 $\epsilon$-greedy 방식으로 선택하게 함
   - [0, 1] 사이의 실수 값을 뽑아 $\epsilon$ 값보다 작으면 랜덤 액션을 하고, 그보다 크면 Q값이 제일 큰 액션을 선택함

## 학습 함수

In [43]:
def train(q, q_target, memory, optimizer):
    for i in range(10):
        s,a,r,s_prime,done_mask = memory.sample(batch_size)

        q_out = q(s)
        q_a = q_out.gather(1,a)
        max_q_prime = q_target(s_prime).max(1)[0].unsqueeze(1)
        target = r + gamma * max_q_prime * done_mask
        loss = F.smooth_l1_loss(q_a, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

- 먼저 리플레이 버퍼에서 미니 배치를 뽑고 해당 데이터를 이용하여 loss 값을 계산함 <br/><br/>
- loss가 계산되면, loss.backward()를 이용하여 그라디어느 계산함 <br/><br/>
- optimizer.step()을 실행하는 순간 Qnet의 파라미터의 업데이트가 진행됨 <br/><br/>
---
- q_a : 실제 선택된 액션의 q 값 <br/><br/>
- max_q_prime를 계산할 땐 q네트워크가 아닌 q_target 네트워크를 호출함
   - q_target 네트워크는 정답을 계산할 때 쓰이는 네트워크로 학습 대상이 아니기 때문에 optimizer에 알려줘야함

## 메인 함수

In [44]:
def main():
    env = gym.make('CartPole-v1')
    q = Qnet()
    q_target = Qnet()
    q_target.load_state_dict(q.state_dict())
    memory = ReplayBuffer()

    print_interval = 20
    score = 0.0
    optimizer = optim.Adam(q.parameters(), lr=learning_rate)
    max_score = 0.0

    for n_epi in range(2000):
        epsilon = max(0.01, 0.08 - 0.01*(n_epi/200))
        # Linear annealing from 8% to 1%
        s, info = env.reset()
        done = False

        while not done:
            a = q.sample_action(torch.from_numpy(s).float(), epsilon)

            s_prime, r, terminated, truncated, info = env.step(a)
            done = terminated or truncated

            # terminated(진짜 종료)만 terminal로 취급하고,
            # truncated(시간제한 등)는 부트스트랩 가능하게 done_mask=1로 두는 게 일반적임
            done_mask = 0.0 if terminated else 1.0

            memory.put((s, a, r/100.0, s_prime, done_mask))
            s = s_prime
            score += r

        if memory.size() > 2000:
            train(q, q_target, memory, optimizer)

        if n_epi % print_interval == 0 and n_epi != 0:
            q_target.load_state_dict(q.state_dict())
            print(f"n_episode :{n_epi}, score : {score/print_interval:.1f}, n_buffer : {memory.size()}, eps : {epsilon*100:.1f}%")
            # torch.save(q_target.state_dict(), 'q_target.pth')
            if epsilon == 0.01 and score > max_score:
                print(f'>>>> save q_target.pth: {score:.1f}')
                torch.save(q_target.state_dict(), 'q_target.pth')
                max_score = score

            score = 0.0

    env.close()

In [45]:
main()

n_episode :20, score : 11.9, n_buffer : 238, eps : 7.9%
n_episode :40, score : 10.9, n_buffer : 456, eps : 7.8%
n_episode :60, score : 11.4, n_buffer : 684, eps : 7.7%
n_episode :80, score : 11.6, n_buffer : 915, eps : 7.6%
n_episode :100, score : 10.9, n_buffer : 1133, eps : 7.5%
n_episode :120, score : 10.7, n_buffer : 1347, eps : 7.4%
n_episode :140, score : 10.9, n_buffer : 1565, eps : 7.3%
n_episode :160, score : 10.7, n_buffer : 1779, eps : 7.2%
n_episode :180, score : 11.2, n_buffer : 2003, eps : 7.1%
n_episode :200, score : 11.7, n_buffer : 2236, eps : 7.0%
n_episode :220, score : 10.5, n_buffer : 2446, eps : 6.9%
n_episode :240, score : 10.8, n_buffer : 2663, eps : 6.8%
n_episode :260, score : 14.0, n_buffer : 2943, eps : 6.7%
n_episode :280, score : 53.0, n_buffer : 4002, eps : 6.6%
n_episode :300, score : 160.8, n_buffer : 7219, eps : 6.5%
n_episode :320, score : 209.2, n_buffer : 11404, eps : 6.4%
n_episode :340, score : 159.7, n_buffer : 14598, eps : 6.3%
n_episode :360, s

## 결과확인

In [52]:
import shutil
import os

env = gym.make('CartPole-v1', render_mode='rgb_array')
q_target = Qnet()
q_target.load_state_dict(torch.load('q_target.pth'))

# Remove existing video directory to ensure a fresh recording
if os.path.exists('./video'):
    shutil.rmtree('./video')

# Recreate the environment wrapper for recording. Set episode_trigger to record every episode.
env = gym.wrappers.RecordVideo(env, './video', episode_trigger=lambda x: True)

In [53]:
s, info = env.reset()
done = False

while not done:
    action = q_target.sample_action(torch.from_numpy(s).float(), 0.0)
    s_prime, r, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    s = s_prime
    print(action, r)
env.close()

0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.

In [54]:
# play recorded video
def show_video():
    mp4list = glob.glob('video/*.mp4')
    if len(mp4list) > 0:
        # Sort by modification time to get the latest video
        mp4 = max(mp4list, key=os.path.getmtime)
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        ipythondisplay.display(HTML(data='''
            <video alt="test" autoplay loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
            </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

In [55]:
show_video()